In [1]:
# 01. EXTRAÇÃO FRAME A FRAME E ERSP (N170)

import os
import glob
import mne
import warnings
import numpy as np
import pandas as pd
from scipy.stats import entropy

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("Iniciando Extração Dinâmica (Frame a Frame) + ERSP...")
print("="*80)

# 1. Diretórios (Mantenha igual ao seu padrão)
DIR_EPOCHS = '../data/processed/epochs/'
DIR_REPORTS = '../reports/'
os.makedirs(DIR_REPORTS, exist_ok=True)
ARQUIVO_SAIDA = os.path.join(DIR_REPORTS, 'features_frame_a_frame_ERSP.csv')

# 2. Parâmetros Base
BANDAS_FREQ = {'Theta': (4.0, 8.0), 'Alpha': (8.0, 13.0), 'Beta': (13.0, 30.0), 'Gamma': (30.0, 40.0)}
CANAIS_PSD = ['Fp1', 'Fp2', 'F7', 'F3', 'F4', 'F8', 'T3', 'C3', 'C4', 'T4', 'T5', 'P3', 'P4', 'T6', 'O1', 'O2']
PARES_ASSIMETRIA = [('F4', 'F3'), ('C4', 'C3'), ('P4', 'P3'), ('O2', 'O1')]
PARES_CONECTIVIDADE = [('F3', 'F4'), ('C3', 'C4'), ('P3', 'P4'), ('O1', 'O2'), ('F3', 'O1'), ('F4', 'O2')]

# 3. Novos Parâmetros para o ERSP (N170 na Frequência)
CANAIS_ERSP = ['O1', 'O2', 'T5', 'T6'] # Expandido para incluir os temporais
JANELA_ERSP = (0.150, 0.200) # Janela estrita de 150ms a 200ms
FREQS_ERSP = np.arange(4, 13, 1) # Pesquisando energia de 4 a 12 Hz (Theta e Alpha)

arquivos_epo = sorted(glob.glob(os.path.join(DIR_EPOCHS, '*-epo.fif')))
todas_features = []

# 4. Varredura dos Pacientes
for caminho_arquivo in arquivos_epo:
    nome_arquivo = os.path.basename(caminho_arquivo)
    sujeito_id = nome_arquivo.split('-epo')[0]
    grupo = 'TEA' if 'TEA' in sujeito_id else 'Control'
    
    epochs = mne.read_epochs(caminho_arquivo, preload=True, verbose=False)
    
    for condicao in epochs.event_id.keys():
        epochs_condicao = epochs[condicao]
        n_epochs = len(epochs_condicao)
        
        if n_epochs == 0:
            continue

        # A. Cálculos Globais (Computa tudo de uma vez para economizar processamento)
        n_tempos = epochs_condicao.get_data().shape[2]
        n_fft = min(256, n_tempos) 
        
        # PSD (Welch) para todas as épocas de uma vez
        psd_data = epochs_condicao.compute_psd(method='welch', fmin=4.0, fmax=40.0, 
                                               n_fft=n_fft, n_overlap=n_fft//2, verbose=False)
        psds, freqs = psd_data.get_data(return_freqs=True) # Shape: (epochs, canais, frequencias)
        
        # B. TFR (Morlet Wavelets) para o ERSP (Apenas se for tarefa de Faces)
        nome_condicao_limpo = condicao
        if condicao in ['FF', 'F']: nome_condicao_limpo = 'Face Feliz'
        elif condicao in ['FN', 'N']: nome_condicao_limpo = 'Face Neutra'
        elif condicao in ['FR', 'R']: nome_condicao_limpo = 'Face Raiva'
        
        tipo_tarefa = 'Task' if 'Face' in nome_condicao_limpo else 'Resting'
        
        tfr_data = None
        if tipo_tarefa == 'Task':
            # Calcula a wavelet época a época (average=False)
            tfr = mne.time_frequency.tfr_morlet(epochs_condicao, freqs=FREQS_ERSP, 
                                                n_cycles=FREQS_ERSP/2.0, return_itc=False, 
                                                average=False, verbose=False)
            tfr_data = tfr.data # Shape: (epochs, canais, freqs, tempos)
            tfr_times = tfr.times

        # C. O NOVO LAÇO: Extração Frame a Frame
        # Em vez de tirar a média, vamos varrer cada época (frame) individualmente
        dados_brutos_epochs = epochs_condicao.get_data()
        
        for epoch_idx in range(n_epochs):
            # Dicionário base para esta linha específica (este frame)
            features_linha = {
                'ID': sujeito_id,
                'Grupo': grupo,
                'Condicao': nome_condicao_limpo,
                'Tipo': tipo_tarefa,
                'Frame_Num': epoch_idx + 1  # Guarda se é a foto 1, 2, 3... até 30
            }
            
            # Pega o PSD apenas desta época
            psd_epoca = psds[epoch_idx, :, :]
            
            # Extração de Potência e Entropia para ESTE frame
            potencia_absoluta = {}
            for ch_idx, canal in enumerate(epochs_condicao.ch_names):
                if canal not in CANAIS_PSD: continue
                potencia_total_canal = 0
                
                # Potência Absoluta
                for banda, (fmin, fmax) in BANDAS_FREQ.items():
                    idx_banda = np.logical_and(freqs >= fmin, freqs <= fmax)
                    potencia = np.sum(psd_epoca[ch_idx, idx_banda])
                    potencia_absoluta[f"{banda}_{canal}"] = potencia
                    potencia_total_canal += potencia
                
                # Potência Relativa
                for banda in BANDAS_FREQ.keys():
                    abs_val = potencia_absoluta[f"{banda}_{canal}"]
                    rel_val = abs_val / potencia_total_canal if potencia_total_canal > 0 else 0
                    features_linha[f"{banda}Rel_{canal}"] = rel_val
                
                # Razão Theta/Beta
                theta_val = potencia_absoluta[f"Theta_{canal}"]
                beta_val = potencia_absoluta[f"Beta_{canal}"]
                features_linha[f"TBR_{canal}"] = theta_val / beta_val if beta_val > 0 else 0
                
                # Entropia Espectral
                psd_norm = psd_epoca[ch_idx, :] / np.sum(psd_epoca[ch_idx, :])
                features_linha[f"Entropia_{canal}"] = entropy(psd_norm)

            # Assimetria para ESTE frame
            for banda in BANDAS_FREQ.keys():
                for ch_dir, ch_esq in PARES_ASSIMETRIA:
                    chave_dir = f"{banda}Rel_{ch_dir}"
                    chave_esq = f"{banda}Rel_{ch_esq}"
                    if chave_dir in features_linha and chave_esq in features_linha:
                        nome_regiao = ch_dir[0] 
                        features_linha[f"Asym_{banda}_{nome_regiao}"] = features_linha[chave_dir] - features_linha[chave_esq]

            # Conectividade para ESTE frame
            dado_epoca_atual = dados_brutos_epochs[epoch_idx, :, :]
            for ch1, ch2 in PARES_CONECTIVIDADE:
                if ch1 in epochs_condicao.ch_names and ch2 in epochs_condicao.ch_names:
                    idx1 = epochs_condicao.ch_names.index(ch1)
                    idx2 = epochs_condicao.ch_names.index(ch2)
                    corr = np.corrcoef(dado_epoca_atual[idx1, :], dado_epoca_atual[idx2, :])[0, 1]
                    features_linha[f"Conn_{ch1}_{ch2}"] = corr

            # Nova Feature: ERSP para ESTE frame
            if tipo_tarefa == 'Task' and tfr_data is not None:
                # Isola os índices de tempo entre 150 e 200ms
                idx_tempo_ersp = np.logical_and(tfr_times >= JANELA_ERSP[0], tfr_times <= JANELA_ERSP[1])
                
                for canal in CANAIS_ERSP:
                    if canal in epochs_condicao.ch_names:
                        ch_idx = epochs_condicao.ch_names.index(canal)
                        # Tira a média da energia (power) apenas na janela de 150-200ms para as frequências selecionadas
                        energia_n170 = np.mean(tfr_data[epoch_idx, ch_idx, :, idx_tempo_ersp])
                        features_linha[f"ERSP_N170_{canal}"] = energia_n170

            todas_features.append(features_linha)

# 5. Exportação
df_features = pd.DataFrame(todas_features)
df_features.fillna(0, inplace=True)
df_features.to_csv(ARQUIVO_SAIDA, index=False)

print(f"Matriz de características DINÂMICA gerada com sucesso.")
print(f"Dimensões do dataset de saída: {df_features.shape[0]} amostras (frames) x {df_features.shape[1]} atributos.")
print("="*80)


Iniciando Extração Dinâmica (Frame a Frame) + ERSP...
Matriz de características DINÂMICA gerada com sucesso.
Dimensões do dataset de saída: 8375 amostras (frames) x 127 atributos.


In [8]:
# 02. PIPELINE DE CLASSIFICAÇÃO FRAME A FRAME (TIME-SERIES)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

print("\n" + "="*80)
print("Iniciando Análise Temporal: XGBoost Frame a Frame (Face Feliz)")
print("="*80)

# 1. Carregamento e Preparação dos Dados
df = pd.read_csv('../reports/features_frame_a_frame_ERSP.csv')

# Filtrar apenas para a condição alvo (Face Feliz)
df_feliz = df[df['Condicao'] == 'Face Feliz'].copy()

# Mapeamento do Target (TEA = 1, Controle = 0)
df_feliz['Target'] = df_feliz['Grupo'].map({'TEA': 1, 'Control': 0})

# Separar metadados e features
colunas_ignorar = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_cols = [c for c in df_feliz.columns if c not in colunas_ignorar]

# 2. Definição do Pipeline de Machine Learning
# Definindo o scale_pos_weight dinamicamente com base na proporção do grupo
n_controles = len(df_feliz[df_feliz['Target'] == 0]['ID'].unique())
n_tea = len(df_feliz[df_feliz['Target'] == 1]['ID'].unique())
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

logo = LeaveOneGroupOut()
frames = sorted(df_feliz['Frame_Num'].unique())
acuracias_por_frame = []

# 3. O Loop Temporal (Máquina do Tempo)
for frame in frames:
    # Isola os dados EXATAMENTE deste frame
    df_frame = df_feliz[df_feliz['Frame_Num'] == frame]
    
    # Se algum paciente perdeu este frame (ex: autoreject), garantimos que os dados batem
    X = df_frame[features_cols].values
    y = df_frame['Target'].values
    grupos_pacientes = df_frame['ID'].values 
    
    previsoes = []
    valores_reais = []
    
    # Validação Cruzada Isolando o Paciente (Prevenção de Data Leakage)
    for train_index, test_index in logo.split(X, y, grupos_pacientes):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Etapa A: Padronização (Ajustada apenas no treino)
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Etapa B: Seleção de Features (ANOVA) - As 15 melhores para ESTE frame
        selector = SelectKBest(score_func=f_classif, k=15)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        # Etapa C: Otimização e Treinamento do Modelo
        modelo = XGBClassifier(
            n_estimators=100,
            max_depth=5,
            scale_pos_weight=peso_classes,
            eval_metric='logloss',
            random_state=42
        )
        modelo.fit(X_train_sel, y_train)
        
        # Etapa D: Previsão no Paciente Oculto
        y_pred = modelo.predict(X_test_sel)
        previsoes.extend(y_pred)
        valores_reais.extend(y_test)
        
    # Calcula a acurácia global deste frame específico
    acc_frame = accuracy_score(valores_reais, previsoes)
    acuracias_por_frame.append(acc_frame)
    print(f"Frame {int(frame):02d}/30 concluído | Acurácia: {acc_frame*100:.2f}%")

# 4. Plotagem da Curva de Atenção/Engajamento Temporal
plt.figure(figsize=(12, 6))
plt.plot(frames, acuracias_por_frame, marker='o', linestyle='-', color='#1f77b4', linewidth=2)
plt.axhline(y=0.8095, color='r', linestyle='--', label='Baseline Qualificação (80.95%)')
plt.axhline(y=0.50, color='gray', linestyle=':', label='Chance Nível (50%)')

plt.title('Dinâmica Temporal de Classificação (XGBoost) - Face Feliz', fontsize=14, pad=15)
plt.xlabel('Época (Frames de 1 segundo)', fontsize=12)
plt.ylabel('Acurácia (LOOCV)', fontsize=12)
plt.xticks(frames)
plt.yticks(np.arange(0.3, 1.0, 0.1))
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()

# Salva o gráfico
caminho_grafico = os.path.join(DIR_REPORTS, 'dinamica_temporal_face_feliz.png')
plt.savefig(caminho_grafico, dpi=300)
plt.show()

print("="*80)
print(f"Análise concluída. Gráfico salvo em: {caminho_grafico}")


Iniciando Análise Temporal: XGBoost Frame a Frame (Face Feliz)


KeyboardInterrupt: 

In [ ]:
# 03. AGREGAÇÃO E TESTE DA NOVA FEATURE ERSP (N170)

import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix
from xgboost import XGBClassifier

print("\n" + "="*80)
print("Iniciando Teste Final: Média das Épocas + Feature ERSP (N170)")
print("="*80)

# 1. Carregamento e Agregação (O Cancelamento do Ruído)
df_frames = pd.read_csv('../reports/features_frame_a_frame_ERSP.csv')
df_feliz = df_frames[df_frames['Condicao'] == 'Face Feliz'].copy()

# Agrupa por ID e tira a média de todas as colunas numéricas (Restaurando a SNR do EEG)
colunas_agrupamento = ['ID', 'Grupo']
features_numericas = [c for c in df_feliz.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num']]

df_agrupado = df_feliz.groupby(colunas_agrupamento)[features_numericas].mean().reset_index()

# 2. Preparação para o Machine Learning
df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values
pacientes = df_agrupado['ID'].values

n_controles = len(df_agrupado[df_agrupado['Target'] == 0])
n_tea = len(df_agrupado[df_agrupado['Target'] == 1])
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

print(f"Dataset restaurado: {len(df_agrupado)} pacientes ({n_controles} Controles, {n_tea} TEA)")

# 3. Pipeline LOOCV Rigoroso
loo = LeaveOneOut()
previsoes = []
valores_reais = []
importancias_acumuladas = np.zeros(len(features_numericas))

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # A. Escalonamento
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # B. Seleção de Features (ANOVA - As 15 melhores)
    selector = SelectKBest(score_func=f_classif, k=15)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    
    # C. Treinamento XGBoost
    modelo = XGBClassifier(
        n_estimators=100,
        max_depth=5,
        scale_pos_weight=peso_classes,
        eval_metric='logloss',
        random_state=42
    )
    modelo.fit(X_train_sel, y_train)
    
    # D. Previsão cega
    y_pred = modelo.predict(X_test_sel)
    previsoes.extend(y_pred)
    valores_reais.extend(y_test)
    
    # E. Rastreamento de Importância
    mask_selecionadas = selector.get_support()
    importancias_parciais = np.zeros(len(features_numericas))
    importancias_parciais[mask_selecionadas] = modelo.feature_importances_
    importancias_acumuladas += importancias_parciais

# 4. Avaliação de Resultados
acc = accuracy_score(valores_reais, previsoes)
cm = confusion_matrix(valores_reais, previsoes)

print(f"\n[RESULTADOS GERAIS]")
print(f"Acurácia Global LOOCV: {acc*100:.2f}%")
print(f"Matriz de Confusão (0=Controle, 1=TEA):\n{cm}")

# 5. Ranking das Features Mais Preditivas
importancias_medias = importancias_acumuladas / len(X)
df_ranking = pd.DataFrame({
    'Feature': features_numericas,
    'Importancia': importancias_medias
}).sort_values(by='Importancia', ascending=False)

print("\n[TOP 10 VARIÁVEIS MAIS IMPORTANTES]")
print(df_ranking[df_ranking['Importancia'] > 0].head(15).to_string(index=False))
print("="*80)


Iniciando Teste Final: Média das Épocas + Feature ERSP (N170)
Dataset restaurado: 42 pacientes (24 Controles, 18 TEA)

[RESULTADOS GERAIS]
Acurácia Global LOOCV: 73.81%
Matriz de Confusão (0=Controle, 1=TEA):
[[19  5]
 [ 6 12]]

[TOP 10 VARIÁVEIS MAIS IMPORTANTES]
     Feature  Importancia
  BetaRel_F7     0.173738
AlphaRel_Fp2     0.159095
 GammaRel_T4     0.115206
  BetaRel_F8     0.112108
 AlphaRel_T5     0.099234
 GammaRel_F7     0.065452
 AlphaRel_P3     0.044717
 AlphaRel_T4     0.039544
 AlphaRel_T3     0.028562
 AlphaRel_F7     0.026716
 BetaRel_Fp2     0.023271
 GammaRel_C4     0.020550
 GammaRel_F8     0.020129
  BetaRel_T4     0.017377
GammaRel_Fp2     0.015830


In [11]:
# 01. APRENDIZADO DE MÁQUINA (CLASSIFICAÇÃO PREDITIVA)

import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings

warnings.filterwarnings("ignore")

print("="*80)
print(" Iniciando avaliação do modelo final (XGBOOST + K=15)")
print("="*80)

# 1. Carregamento e Isolamento da Tarefa
caminho_csv = '../reports/tabela_features_eeg_completa.csv'
df = pd.read_csv(caminho_csv)

SEED = 97
cv_strategy = LeaveOneOut() # LOOCV (N=42)

condicoes_busca = {
    'Face Feliz': ['Face Feliz'],
    'Face Neutra': ['Face Neutra'],
    'Face Raiva': ['Face Raiva']
}

# 2. Pipeline Final
pipeline_classificacao = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', RobustScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=15)),
    ('clf', XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, 
                          eval_metric='logloss', random_state=SEED))
])

# 3. Validação e Extração de Métricas
for nome_condicao, lista_triggers in condicoes_busca.items():
    print(f"\n" + "-"*60)
    print(f"ESTÍMULO ANALISADO: {nome_condicao.upper()}")
    print("-" * 60)
    
    df_f = df[df['Condicao'].isin(lista_triggers)].copy()
    
    if df_f.empty:
        print(f"Atenção: Sem dados para {nome_condicao}.")
        continue

    # Garantia de integridade amostral (N=42)
    df_f = df_f.groupby(['ID', 'Grupo']).mean(numeric_only=True).reset_index()
    print(f"Amostra consolidada: {df_f.shape[0]} sujeitos.\n")

    y = df_f['Grupo'].apply(lambda x: 1 if 'TEA' in x else 0).values
    X = df_f.drop(columns=['ID', 'Grupo', 'Condicao', 'Tipo'], errors='ignore')

    # Predição Cega (LOOCV)
    y_pred = cross_val_predict(pipeline_classificacao, X, y, cv=cv_strategy)

    acc = accuracy_score(y, y_pred)
    cm = confusion_matrix(y, y_pred)

    print(f" ACURÁCIA GLOBAL DO SISTEMA: {acc:.2%}\n")

    print("Matriz de Confusão:")
    print(f"                   Predito Controle (0) | Predito TEA (1)")
    print(f"Real Controle (0) |        {cm[0,0]:02d}           |        {cm[0,1]:02d}")
    print(f"Real TEA (1)      |        {cm[1,0]:02d}           |        {cm[1,1]:02d}\n")

    print("Relatório de Desempenho (Precision, Recall, F1-Score):")
    report = classification_report(y, y_pred, target_names=['Controle', 'TEA'], digits=3)
    print(report)

print("="*80)

 Iniciando avaliação do modelo final (XGBOOST + K=15)

------------------------------------------------------------
ESTÍMULO ANALISADO: FACE FELIZ
------------------------------------------------------------
Amostra consolidada: 42 sujeitos.

 ACURÁCIA GLOBAL DO SISTEMA: 80.95%

Matriz de Confusão:
                   Predito Controle (0) | Predito TEA (1)
Real Controle (0) |        21           |        03
Real TEA (1)      |        05           |        13

Relatório de Desempenho (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

    Controle      0.808     0.875     0.840        24
         TEA      0.812     0.722     0.765        18

    accuracy                          0.810        42
   macro avg      0.810     0.799     0.802        42
weighted avg      0.810     0.810     0.808        42


------------------------------------------------------------
ESTÍMULO ANALISADO: FACE NEUTRA
------------------------------------------------------------


In [1]:
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, precision_score, f1_score
from xgboost import XGBClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings

warnings.filterwarnings("ignore")

print(" Calculando métricas clínicas globais para todas as Faces...\n")

# Carregamento dos Dados
caminho_csv = '../reports/tabela_features_eeg_completa.csv'
try:
    df = pd.read_csv(caminho_csv)
except FileNotFoundError:
    df = pd.read_csv('reports/tabela_features_eeg_completa.csv')

condicoes = {
    'FF': 'Face Feliz',
    'FN': 'Face Neutra',
    'FR': 'Face Raiva'
}

# Cabeçalho da Tabela
print("Tabela - Acurácia Global do modelo XGBoost para os dados de EEG")
print("-" * 75)
print(f"{'EEG':<5} | {'Acurácia':<10} | {'Sensibilidade':<15} | {'Precisão':<10} | {'F1-Score':<10} | {'AUC-ROC':<10}")
print("-" * 75)

for sigla, condicao in condicoes.items():
    df_cond = df[df['Condicao'] == condicao].copy()
    
    if df_cond.empty:
        print(f"{sigla:<5} | Sem dados para esta condição")
        continue

    # Agrupamento e definição de variáveis
    df_cond = df_cond.groupby(['ID', 'Grupo']).mean(numeric_only=True).reset_index()
    y = df_cond['Grupo'].apply(lambda x: 1 if 'TEA' in x else 0).values
    X = df_cond.drop(columns=['ID', 'Grupo', 'Condicao', 'Tipo'], errors='ignore')

    # Pipeline
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', RobustScaler()),
        ('selector', SelectKBest(score_func=f_classif, k=15)),
        ('clf', XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.1, 
                              eval_metric='logloss', random_state=97))
    ])

    # Predições LOOCV
    cv = LeaveOneOut()
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)
    y_probs = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')[:, 1]

    # Cálculos Matemáticos
    cm = confusion_matrix(y, y_pred)
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y, y_pred) * 100
    sens = (tp / (tp + fn) * 100) if (tp + fn) > 0 else 0
    prec = (precision_score(y, y_pred, zero_division=0) * 100)
    f1 = (f1_score(y, y_pred, zero_division=0) * 100)
    
    # Transformando AUC para formato de porcentagem
    auc = (roc_auc_score(y, y_probs) * 100) 

    # Impressão da linha formatada
    print(f"{sigla:<5} | {acc:>6.2f}%   | {sens:>12.2f}%  | {prec:>7.2f}%  | {f1:>7.2f}%  | {auc:>7.2f}%")

print("-" * 75)

 Calculando métricas clínicas globais para todas as Faces...

Tabela - Acurácia Global do modelo XGBoost para os dados de EEG
---------------------------------------------------------------------------
EEG   | Acurácia   | Sensibilidade   | Precisão   | F1-Score   | AUC-ROC   
---------------------------------------------------------------------------
FF    |  83.33%   |        77.78%  |   82.35%  |   80.00%  |   74.07%
FN    |  57.14%   |        50.00%  |   50.00%  |   50.00%  |   63.66%
FR    |  64.29%   |        44.44%  |   61.54%  |   51.61%  |   68.75%
---------------------------------------------------------------------------


In [2]:
# 01. OTIMIZAÇÃO DE HIPERPARÂMETROS E MODELOS (GRID SEARCH LOOCV)

import pandas as pd
import numpy as np
import os
import time
from itertools import product
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("Iniciando Batalha de Modelos: Grid Search + LOOCV")
print("="*80)

# 1. Carregamento e Preparação dos Dados (Mesmo padrão que validamos)
DIR_REPORTS = '../reports/'
df_frames = pd.read_csv(os.path.join(DIR_REPORTS, 'features_frame_a_frame_ERSP.csv'))
df_feliz = df_frames[df_frames['Condicao'] == 'Face Feliz'].copy()

# Restauração do Sinal (Agregando por Paciente)
colunas_agrupamento = ['ID', 'Grupo']
features_numericas = [c for c in df_feliz.columns if c not in ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num']]
df_agrupado = df_feliz.groupby(colunas_agrupamento)[features_numericas].mean().reset_index()

df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})
X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values

# Cálculo do peso para desbalanceamento (XGBoost)
n_controles = len(df_agrupado[df_agrupado['Target'] == 0])
n_tea = len(df_agrupado[df_agrupado['Target'] == 1])
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

# 2. Definição da Grade de Busca (O que você mapeou)
k_valores = [5, 8, 10, 12, 15, 18, 20]
profundidades = [3, 4]
estimadores = [50, 100, 150, 200]
modelos = ['XGBoost', 'RandomForest']

# O itertools.product cria todas as 112 combinações instantaneamente
todas_combinacoes = list(product(modelos, k_valores, profundidades, estimadores))
total_comb = len(todas_combinacoes)

print(f"Dataset carregado: {len(X)} pacientes.")
print(f"Iniciando {total_comb} cenários. Isso equivale a {total_comb * len(X)} treinamentos...")
print("Por favor, aguarde. Isso pode levar alguns minutos.\n")

resultados = []
loo = LeaveOneOut()
tempo_inicio = time.time()

# 3. O Loop Gigante
for idx, (nome_modelo, k, depth, est) in enumerate(todas_combinacoes, 1):
    previsoes = []
    valores_reais = []
    
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Escalonamento Rigoroso
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Seleção Dinâmica de Variáveis (K varia a cada rodada)
        selector = SelectKBest(score_func=f_classif, k=k)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        # Configuração do Motor
        if nome_modelo == 'XGBoost':
            modelo = XGBClassifier(n_estimators=est, max_depth=depth, scale_pos_weight=peso_classes, 
                                   eval_metric='logloss', random_state=42)
        else: # Random Forest
            modelo = RandomForestClassifier(n_estimators=est, max_depth=depth, class_weight='balanced', 
                                            random_state=42)
            
        modelo.fit(X_train_sel, y_train)
        y_pred = modelo.predict(X_test_sel)
        
        previsoes.extend(y_pred)
        valores_reais.extend(y_test)
        
    # Acurácia do cenário atual
    acc = accuracy_score(valores_reais, previsoes)
    resultados.append({
        'Modelo': nome_modelo,
        'K_Features': k,
        'Profundidade': depth,
        'Estimadores': est,
        'Acurácia': acc
    })
    
    # Print de progresso a cada 20 cenários
    if idx % 20 == 0 or idx == total_comb:
        print(f"[{idx}/{total_comb}] Cenários testados...")

# 4. Avaliação e Ranking
df_resultados = pd.DataFrame(resultados)
df_ranking = df_resultados.sort_values(by=['Acurácia', 'Estimadores'], ascending=[False, True]).reset_index(drop=True)

tempo_total = (time.time() - tempo_inicio) / 60

print("\n" + "="*80)
print(f"Busca finalizada em {tempo_total:.1f} minutos!")
print("="*80)
print("\n🏆 [TOP 10 MELHORES COMBINAÇÕES] 🏆\n")
print(df_ranking.head(10).to_string(index=False))

# Salva o log completo para sua defesa
caminho_csv = os.path.join(DIR_REPORTS, 'grid_search_resultados_completos.csv')
df_ranking.to_csv(caminho_csv, index=False)
print(f"\nTabela completa com as {total_comb} combinações salva em: {caminho_csv}")
print("="*80)


Iniciando Batalha de Modelos: Grid Search + LOOCV
Dataset carregado: 42 pacientes.
Iniciando 112 cenários. Isso equivale a 4704 treinamentos...
Por favor, aguarde. Isso pode levar alguns minutos.

[20/112] Cenários testados...
[40/112] Cenários testados...
[60/112] Cenários testados...
[80/112] Cenários testados...
[100/112] Cenários testados...
[112/112] Cenários testados...

Busca finalizada em 5.6 minutos!

🏆 [TOP 10 MELHORES COMBINAÇÕES] 🏆

      Modelo  K_Features  Profundidade  Estimadores  Acurácia
     XGBoost          20             3           50  0.785714
     XGBoost          20             4           50  0.785714
     XGBoost          20             3          100  0.785714
     XGBoost          20             4          100  0.785714
     XGBoost          20             3          150  0.785714
     XGBoost          20             4          150  0.785714
     XGBoost          20             3          200  0.785714
     XGBoost          20             4          200  0

In [5]:
# 01. OTIMIZAÇÃO DE HIPERPARÂMETROS E MODELOS (GRID SEARCH LOOCV) - BASE CORRETA

import pandas as pd
import numpy as np
import os
import time
from itertools import product
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("Iniciando Batalha de Modelos: Grid Search + LOOCV (Base Original)")
print("="*80)

# 1. Carregamento e Preparação Inteligente dos Dados
DIR_REPORTS = '../reports/'

# ==============================================================================
# ATENÇÃO: Substitua pelo nome exato do arquivo CSV usado no seu Notebook 03
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
# ==============================================================================

caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

if not os.path.exists(caminho_arquivo):
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}.")

df = pd.read_csv(caminho_arquivo)

# Lógica inteligente para preparar a matriz (caso tenha Condicao ou já esteja pronta)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

# Remove colunas de metadados para isolar as numéricas
colunas_ignoradas = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_numericas = [c for c in df.columns if c not in colunas_ignoradas]

# Agrupa por paciente (tirando a média) caso existam múltiplas linhas por ID
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

# Configuração do Target (Alvo)
if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values

# Cálculo do peso para desbalanceamento (XGBoost)
n_controles = len(df_agrupado[df_agrupado['Target'] == 0])
n_tea = len(df_agrupado[df_agrupado['Target'] == 1])
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

# 2. Definição da Grade de Busca
k_valores = [5, 8, 10, 12, 15, 18, 20]
profundidades = [3, 4]
estimadores = [50, 100, 150, 200]
modelos = ['XGBoost', 'RandomForest']

todas_combinacoes = list(product(modelos, k_valores, profundidades, estimadores))
total_comb = len(todas_combinacoes)

print(f"Dataset carregado e validado: {len(X)} pacientes.")
print(f"Iniciando {total_comb} cenários. Isso equivale a {total_comb * len(X)} treinamentos...")
print("Por favor, aguarde. O loop está em execução...\n")

resultados = []
loo = LeaveOneOut()
tempo_inicio = time.time()

# 3. O Loop Gigante
for idx, (nome_modelo, k, depth, est) in enumerate(todas_combinacoes, 1):
    previsoes = []
    valores_reais = []
    
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Escalonamento
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Seleção Dinâmica de Variáveis
        # Trava de segurança: k não pode ser maior que o número de variáveis disponíveis
        k_real = min(k, X_train_scaled.shape[1]) 
        selector = SelectKBest(score_func=f_classif, k=k_real)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        # Configuração do Motor
        if nome_modelo == 'XGBoost':
            modelo = XGBClassifier(n_estimators=est, max_depth=depth, scale_pos_weight=peso_classes, 
                                   eval_metric='logloss', random_state=42, n_jobs=-1)
        else: # Random Forest
            modelo = RandomForestClassifier(n_estimators=est, max_depth=depth, class_weight='balanced', 
                                            random_state=42, n_jobs=-1)
            
        modelo.fit(X_train_sel, y_train)
        y_pred = modelo.predict(X_test_sel)
        
        previsoes.extend(y_pred)
        valores_reais.extend(y_test)
        
    # Acurácia do cenário atual
    acc = accuracy_score(valores_reais, previsoes)
    resultados.append({
        'Modelo': nome_modelo,
        'K_Features': k_real,
        'Profundidade': depth,
        'Estimadores': est,
        'Acurácia': acc
    })
    
    # Print de progresso a cada 20 cenários para você não achar que travou
    if idx % 20 == 0 or idx == total_comb:
        print(f"[{idx}/{total_comb}] Cenários concluídos...")

# 4. Avaliação e Ranking Final
df_resultados = pd.DataFrame(resultados)
df_ranking = df_resultados.sort_values(by=['Acurácia', 'Estimadores'], ascending=[False, True]).reset_index(drop=True)

tempo_total = (time.time() - tempo_inicio) / 60

print("\n" + "="*80)
print(f"Busca finalizada com sucesso em {tempo_total:.1f} minutos!")
print("="*80)
print("\n🏆 [TOP 10 MELHORES COMBINAÇÕES] 🏆\n")
print(df_ranking.head(10).to_string(index=False))

caminho_csv = os.path.join(DIR_REPORTS, 'grid_search_resultados_completos.csv')
df_ranking.to_csv(caminho_csv, index=False)
print(f"\nTabela completa com as {total_comb} combinações salva em: {caminho_csv}")
print("="*80)


Iniciando Batalha de Modelos: Grid Search + LOOCV (Base Original)
Dataset carregado e validado: 42 pacientes.
Iniciando 112 cenários. Isso equivale a 4704 treinamentos...
Por favor, aguarde. O loop está em execução...

[20/112] Cenários concluídos...
[40/112] Cenários concluídos...
[60/112] Cenários concluídos...
[80/112] Cenários concluídos...
[100/112] Cenários concluídos...
[112/112] Cenários concluídos...

Busca finalizada com sucesso em 8.0 minutos!

🏆 [TOP 10 MELHORES COMBINAÇÕES] 🏆

      Modelo  K_Features  Profundidade  Estimadores  Acurácia
     XGBoost           5             3          100  0.833333
RandomForest           8             4          150  0.833333
RandomForest           8             4          200  0.833333
     XGBoost           8             3           50  0.809524
     XGBoost           8             4           50  0.809524
RandomForest           8             3           50  0.809524
RandomForest           8             4           50  0.809524
RandomFo

In [7]:
# 02. IDENTIFICAÇÃO DAS 5 VARIÁVEIS DE OURO (K=5)

import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif

print("\n" + "="*80)
print("Extraindo as Top 5 Variáveis (Assinatura Eletrofisiológica)")
print("="*80)

DIR_REPORTS = '../reports/'

# ==============================================================================
# ATENÇÃO: Use O MESMO arquivo que você usou no Grid Search
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
# ==============================================================================

caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)
df = pd.read_csv(caminho_arquivo)

# Preparação inteligente (mesma do script anterior)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

colunas_ignoradas = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_numericas = [c for c in df.columns if c not in colunas_ignoradas]

if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values

# 1. Escalonamento idêntico ao do Grid Search
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# 2. Seleção das 5 melhores variáveis
selector = SelectKBest(score_func=f_classif, k=5)
selector.fit(X_scaled, y)

# 3. Mapeamento dos nomes e pontuações
mascara_selecionadas = selector.get_support()
variaveis_top5 = np.array(features_numericas)[mascara_selecionadas]
pontuacoes_anova = selector.scores_[mascara_selecionadas]
p_valores = selector.pvalues_[mascara_selecionadas]

# 4. Exibição do Ranking
df_top5 = pd.DataFrame({
    'Feature (Região/Banda)': variaveis_top5,
    'F-Score (Força de Separação)': pontuacoes_anova,
    'P-Valor': p_valores
})

# Ordenando da mais forte para a menos forte
df_top5 = df_top5.sort_values(by='F-Score (Força de Separação)', ascending=False).reset_index(drop=True)

print("\nAs 5 Variáveis que garantiram os 83% de Acurácia:\n")
print(df_top5.to_string())
print("\n" + "="*80)


Extraindo as Top 5 Variáveis (Assinatura Eletrofisiológica)

As 5 Variáveis que garantiram os 83% de Acurácia:

  Feature (Região/Banda)  F-Score (Força de Separação)   P-Valor
0            AlphaRel_P3                     12.919824  0.000882
1           AlphaRel_Fp2                     12.717112  0.000957
2             BetaRel_F7                     11.892270  0.001341
3            AlphaRel_F8                     11.464412  0.001602
4            GammaRel_F7                     10.217717  0.002717



In [ ]:
# 01. OTIMIZAÇÃO DE HIPERPARÂMETROS E MODELOS (GRID SEARCH LOOCV) - BASE CORRETA

import pandas as pd
import numpy as np
import os
import time
from itertools import product
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings

warnings.filterwarnings('ignore')

print("\n" + "="*80)
print("Iniciando Batalha de Modelos: Grid Search + LOOCV (Base Original)")
print("="*80)

# 1. Carregamento e Preparação Inteligente dos Dados
DIR_REPORTS = '../reports/'

# ==============================================================================
# ATENÇÃO: Substitua pelo nome exato do arquivo CSV usado no seu Notebook 03
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
# ==============================================================================

caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)

if not os.path.exists(caminho_arquivo):
    raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}.")

df = pd.read_csv(caminho_arquivo)

# Lógica inteligente para preparar a matriz (caso tenha Condicao ou já esteja pronta)
if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

# Remove colunas de metadados para isolar as numéricas
colunas_ignoradas = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_numericas = [c for c in df.columns if c not in colunas_ignoradas]

# Agrupa por paciente (tirando a média) caso existam múltiplas linhas por ID
if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

# Configuração do Target (Alvo)
if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

X = df_agrupado[features_numericas].values
y = df_agrupado['Target'].values

# Cálculo do peso para desbalanceamento (XGBoost)
n_controles = len(df_agrupado[df_agrupado['Target'] == 0])
n_tea = len(df_agrupado[df_agrupado['Target'] == 1])
peso_classes = n_controles / n_tea if n_tea > 0 else 1.0

# 2. Definição da Grade de Busca
k_valores = [5, 8, 10, 12, 15, 18, 20]
profundidades = [3, 4]
estimadores = [50, 100, 150, 200]
modelos = ['XGBoost', 'RandomForest']

todas_combinacoes = list(product(modelos, k_valores, profundidades, estimadores))
total_comb = len(todas_combinacoes)

print(f"Dataset carregado e validado: {len(X)} pacientes.")
print(f"Iniciando {total_comb} cenários. Isso equivale a {total_comb * len(X)} treinamentos...")
print("Por favor, aguarde. O loop está em execução...\n")

resultados = []
loo = LeaveOneOut()
tempo_inicio = time.time()

# 3. O Loop Gigante
for idx, (nome_modelo, k, depth, est) in enumerate(todas_combinacoes, 1):
    previsoes = []
    valores_reais = []
    
    for train_index, test_index in loo.split(X):
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        # Escalonamento
        scaler = RobustScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Seleção Dinâmica de Variáveis
        # Trava de segurança: k não pode ser maior que o número de variáveis disponíveis
        k_real = min(k, X_train_scaled.shape[1]) 
        selector = SelectKBest(score_func=f_classif, k=k_real)
        X_train_sel = selector.fit_transform(X_train_scaled, y_train)
        X_test_sel = selector.transform(X_test_scaled)
        
        # Configuração do Motor
        if nome_modelo == 'XGBoost':
            modelo = XGBClassifier(n_estimators=est, max_depth=depth, scale_pos_weight=peso_classes, 
                                   eval_metric='logloss', random_state=97, n_jobs=-1)
        else: # Random Forest
            modelo = RandomForestClassifier(n_estimators=est, max_depth=depth, class_weight='balanced', 
                                            random_state=97, n_jobs=-1)
            
        modelo.fit(X_train_sel, y_train)
        y_pred = modelo.predict(X_test_sel)
        
        previsoes.extend(y_pred)
        valores_reais.extend(y_test)
        
    # Acurácia do cenário atual
    acc = accuracy_score(valores_reais, previsoes)
    resultados.append({
        'Modelo': nome_modelo,
        'K_Features': k_real,
        'Profundidade': depth,
        'Estimadores': est,
        'Acurácia': acc
    })
    
    # Print de progresso a cada 20 cenários para você não achar que travou
    if idx % 20 == 0 or idx == total_comb:
        print(f"[{idx}/{total_comb}] Cenários concluídos...")

# 4. Avaliação e Ranking Final
df_resultados = pd.DataFrame(resultados)
df_ranking = df_resultados.sort_values(by=['Acurácia', 'Estimadores'], ascending=[False, True]).reset_index(drop=True)

tempo_total = (time.time() - tempo_inicio) / 60

print("\n" + "="*80)
print(f"Busca finalizada com sucesso em {tempo_total:.1f} minutos!")
print("="*80)
print("\n🏆 [TOP 10 MELHORES COMBINAÇÕES] 🏆\n")
print(df_ranking.head(10).to_string(index=False))

caminho_csv = os.path.join(DIR_REPORTS, 'grid_search_resultados_completos.csv')
df_ranking.to_csv(caminho_csv, index=False)
print(f"\nTabela completa com as {total_comb} combinações salva em: {caminho_csv}")
print("="*80)

In [9]:
# 03. VARREDURA DE AMPLO ESPECTRO COM LAZYCLASSIFIER

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Importando o LazyPredict
from lazypredict.Supervised import LazyClassifier

print("\n" + "="*80)
print("Iniciando Radar de Algoritmos (LazyClassifier) com as 5 Features de Ouro")
print("="*80)

DIR_REPORTS = '../reports/'

# ==============================================================================
# ATENÇÃO: Use O MESMO arquivo original dos testes anteriores
ARQUIVO_MATRIZ_ORIGINAL = 'tabela_features_eeg_completa.csv' 
# ==============================================================================

caminho_arquivo = os.path.join(DIR_REPORTS, ARQUIVO_MATRIZ_ORIGINAL)
df = pd.read_csv(caminho_arquivo)

if 'Condicao' in df.columns:
    df = df[df['Condicao'] == 'Face Feliz'].copy()

colunas_ignoradas = ['ID', 'Grupo', 'Condicao', 'Tipo', 'Frame_Num', 'Target']
features_numericas = [c for c in df.columns if c not in colunas_ignoradas]

if 'ID' in df.columns and df['ID'].duplicated().any():
    df_agrupado = df.groupby(['ID', 'Grupo'])[features_numericas].mean().reset_index()
else:
    df_agrupado = df.copy()

if 'Target' not in df_agrupado.columns:
    df_agrupado['Target'] = df_agrupado['Grupo'].map({'TEA': 1, 'Control': 0})

# Isolando APENAS as 5 variáveis mapeadas anteriormente
features_ouro = ['AlphaRel_P3', 'AlphaRel_Fp2', 'BetaRel_F7', 'AlphaRel_F8', 'GammaRel_F7']

# Verificação de segurança
for feature in features_ouro:
    if feature not in df_agrupado.columns:
        raise ValueError(f"A feature {feature} não foi encontrada na matriz.")

X = df_agrupado[features_ouro].values
y = df_agrupado['Target'].values

# Criando a divisão de Treino e Teste (Stratified para manter a proporção TEA/Controle)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

print(f"Treinando com {len(X_train)} pacientes, Testando com {len(X_test)} pacientes.")
print("Iniciando varredura (isso levará apenas alguns segundos)...\n")

# Instanciando e rodando o LazyClassifier
# O LazyClassifier já faz o escalonamento (StandardScaler) internamente
clf = LazyClassifier(verbose=0, ignore_warnings=True, custom_metric=None, random_state=42)
modelos, predicoes = clf.fit(X_train, X_test, y_train, y_test)

print("\n🏆 [RANKING DO LAZYCLASSIFIER] 🏆\n")
print(modelos.head(15).to_string())
print("\n" + "="*80)


Iniciando Radar de Algoritmos (LazyClassifier) com as 5 Features de Ouro
Treinando com 29 pacientes, Testando com 13 pacientes.
Iniciando varredura (isso levará apenas alguns segundos)...


🏆 [RANKING DO LAZYCLASSIFIER] 🏆

                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                        
KNeighborsClassifier         0.923077           0.916667  0.916667  0.922145   0.932692  0.923077    0.026311
XGBClassifier                0.923077           0.916667  0.857143  0.922145   0.932692  0.923077    0.043990
RandomForestClassifier       0.923077           0.916667  0.833333  0.922145   0.932692  0.923077    0.171641
DecisionTreeClassifier       0.846154           0.845238  0.845238  0.846154   0.846154  0.846154    0.017592
GaussianNB                   0.846154           0.845238  0.833333  0.846154   0.846154  0.846154    0.019692
RidgeC